# Performance Evaluation of Distributed Log Processing Using Apache Spark

## Project Objective

The objective of this project is to compare the performance of sequential Python and Apache Spark for processing web-server log data.

The experiment evaluates:

- Execution Time
- Throughput
- Speedup
- Scalability
- Effect of Spark Parallelism

Dataset sizes used:

- 10,000 records
- 100,000 records
- 500,000 records
- 1,000,000 records
- 5,000,000 records

## Experimental Approach

The same log-processing operations are performed using:

1. Sequential Python
2. Apache Spark

The operations include:

- Total request count
- HTTP status-code count
- 4xx client-error count
- 5xx server-error count
- URL-wise request count

Each performance experiment is executed five times.

Spark parallelism is also evaluated using:

- local[1]
- local[2]
- local[4]
- local[8]

In [1]:
import os
import csv
import time
import random
import math

from collections import Counter
from datetime import datetime, timedelta

# Java configuration required by PySpark
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"

print("Environment setup completed.")
print("CPU logical processors:", os.cpu_count())

Environment setup completed.
CPU logical processors: 8


Apache Spark is initialized in local mode using two processing threads.

The `local[2]` configuration is used for the main dataset-size comparison between Sequential Python and Apache Spark.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("DistributedLogProcessing")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark started successfully.")
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

C:\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark started successfully.
Spark version: 4.2.0
Spark master: local[2]


## Web Server Log Dataset Generation

For the experiment, synthetic web-server log data is generated.

Each log record contains:

- Timestamp
- HTTP Method
- URL
- HTTP Status Code
- Response Size

Five different dataset sizes are generated to study scalability.

In [3]:
urls = ["/", "/home", "/login", "/products", "/cart", "/checkout"]
methods = ["GET", "POST"]
status_codes = [200, 200, 200, 201, 404, 500]

# Fixed seed makes the generated dataset reproducible
random.seed(42)

def generate_logs(number_of_records, file_name):
    
    start_time = datetime(2026, 1, 1, 10, 0, 0)

    with open(file_name, "w", newline="", encoding="utf-8") as file:
        
        writer = csv.writer(file)

        writer.writerow([
            "timestamp",
            "method",
            "url",
            "status",
            "response_size"
        ])

        for i in range(number_of_records):

            timestamp = start_time + timedelta(seconds=i)

            writer.writerow([
                timestamp,
                random.choice(methods),
                random.choice(urls),
                random.choice(status_codes),
                random.randint(100, 5000)
            ])

    print(f"{number_of_records:,} log records created successfully!")

In [4]:
generate_logs(10_000, "logs_10k.csv")
generate_logs(100_000, "logs_100k.csv")
generate_logs(500_000, "logs_500k.csv")
generate_logs(1_000_000, "logs_1m.csv")
generate_logs(5_000_000, "logs_5m.csv")

10,000 log records created successfully!
100,000 log records created successfully!
500,000 log records created successfully!
1,000,000 log records created successfully!
5,000,000 log records created successfully!


## Sequential Python Log Processing

This implementation processes the web-server log file sequentially using Python.

The following operations are performed:

- Count total requests
- Count requests by HTTP status code
- Count 4xx client errors
- Count 5xx server errors
- Count requests for each URL

In [6]:
def process_logs_sequential(file_name):

    total_requests = 0
    status_counts = Counter()
    url_counts = Counter()

    client_errors = 0
    server_errors = 0

    with open(file_name, "r", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:

            total_requests += 1

            status = int(row["status"])
            url = row["url"]

            status_counts[status] += 1
            url_counts[url] += 1

            if 400 <= status < 500:
                client_errors += 1

            elif 500 <= status < 600:
                server_errors += 1

    return (
        total_requests,
        status_counts,
        client_errors,
        server_errors,
        url_counts
    )

In [7]:
python_result = process_logs_sequential("logs_10k.csv")

print("Total Requests:", python_result[0])
print("Status Counts:", python_result[1])
print("4xx Client Errors:", python_result[2])
print("5xx Server Errors:", python_result[3])
print("URL Counts:", python_result[4])

Total Requests: 10000
Status Counts: Counter({200: 4927, 201: 1713, 404: 1686, 500: 1674})
4xx Client Errors: 1686
5xx Server Errors: 1674
URL Counts: Counter({'/checkout': 1700, '/products': 1691, '/': 1681, '/home': 1670, '/login': 1649, '/cart': 1609})


The same web-server log-processing operations are now implemented using Apache Spark.

Spark loads the log data into a DataFrame and performs:

- Total request count
- HTTP status-code count
- 4xx client-error count
- 5xx server-error count
- URL-wise request count

In [8]:
from pyspark.sql.functions import col

def process_logs_spark(file_name):

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_name)
    )

    total_requests = df.count()

    status_counts = (
        df.groupBy("status")
        .count()
        .orderBy("status")
        .collect()
    )

    client_errors = df.filter(
        (col("status") >= 400) &
        (col("status") < 500)
    ).count()

    server_errors = df.filter(
        (col("status") >= 500) &
        (col("status") < 600)
    ).count()

    url_counts = (
        df.groupBy("url")
        .count()
        .collect()
    )

    return (
        total_requests,
        status_counts,
        client_errors,
        server_errors,
        url_counts
    )

In [9]:
spark_result = process_logs_spark("logs_10k.csv")

print("Total Requests:", spark_result[0])
print("Status Counts:", spark_result[1])
print("4xx Client Errors:", spark_result[2])
print("5xx Server Errors:", spark_result[3])
print("URL Counts:", spark_result[4])

Total Requests: 10000
Status Counts: [Row(status=200, count=4927), Row(status=201, count=1713), Row(status=404, count=1686), Row(status=500, count=1674)]
4xx Client Errors: 1686
5xx Server Errors: 1674
URL Counts: [Row(url='/cart', count=1609), Row(url='/products', count=1691), Row(url='/login', count=1649), Row(url='/home', count=1670), Row(url='/checkout', count=1700), Row(url='/', count=1681)]


## Correctness Verification

Before comparing performance, the outputs of Sequential Python and Apache Spark are verified.

Both implementations should produce the same:

- Total request count
- HTTP status-code counts
- 4xx error count
- 5xx error count
- URL-wise request counts

This ensures that the performance comparison is based on equivalent log-processing operations.

In [10]:
# Convert Spark results into Python dictionaries

spark_status_counts = {
    row["status"]: row["count"]
    for row in spark_result[1]
}

spark_url_counts = {
    row["url"]: row["count"]
    for row in spark_result[4]
}

# Compare Python and Spark results

total_match = python_result[0] == spark_result[0]

status_match = dict(python_result[1]) == spark_status_counts

client_error_match = python_result[2] == spark_result[2]

server_error_match = python_result[3] == spark_result[3]

url_match = dict(python_result[4]) == spark_url_counts


print("Total Request Count Match:", total_match)
print("Status Counts Match:", status_match)
print("4xx Error Count Match:", client_error_match)
print("5xx Error Count Match:", server_error_match)
print("URL Counts Match:", url_match)

Total Request Count Match: True
Status Counts Match: True
4xx Error Count Match: True
5xx Error Count Match: True
URL Counts Match: True


## Performance Measurement

After verifying correctness, the performance of Sequential Python and Apache Spark is measured.

The following metrics are calculated:

- Execution Time: Total time required to process the log dataset.
- Throughput: Number of log records processed per second.

Each experiment will be executed five times to reduce the effect of normal timing variations.

In [11]:
def experiment_sequential(file_name):

    start_time = time.perf_counter()

    total_requests = 0
    status_counts = Counter()
    url_counts = Counter()

    client_errors = 0
    server_errors = 0

    with open(file_name, "r", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:

            total_requests += 1

            status = int(row["status"])
            url = row["url"]

            status_counts[status] += 1
            url_counts[url] += 1

            if 400 <= status < 500:
                client_errors += 1

            elif 500 <= status < 600:
                server_errors += 1

    execution_time = time.perf_counter() - start_time

    throughput = total_requests / execution_time

    return execution_time, throughput

In [12]:
def experiment_spark(file_name):

    start_time = time.perf_counter()

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_name)
    )

    total_requests = df.count()

    df.groupBy("status").count().collect()

    df.filter(
        (col("status") >= 400) &
        (col("status") < 500)
    ).count()

    df.filter(
        (col("status") >= 500) &
        (col("status") < 600)
    ).count()

    df.groupBy("url").count().collect()

    execution_time = time.perf_counter() - start_time

    throughput = total_requests / execution_time

    return execution_time, throughput

In [13]:
python_10k_results = []

for run in range(1, 6):

    execution_time, throughput = experiment_sequential("logs_10k.csv")

    python_10k_results.append((execution_time, throughput))

    print(
        f"Run {run}: "
        f"Time = {execution_time:.6f} seconds, "
        f"Throughput = {throughput:.2f} records/second"
    )

Run 1: Time = 0.050288 seconds, Throughput = 198855.78 records/second
Run 2: Time = 0.053839 seconds, Throughput = 185740.69 records/second
Run 3: Time = 0.049488 seconds, Throughput = 202068.78 records/second
Run 4: Time = 0.054746 seconds, Throughput = 182663.42 records/second
Run 5: Time = 0.061550 seconds, Throughput = 162468.48 records/second


In [14]:
spark_10k_results = []

for run in range(1, 6):

    execution_time, throughput = experiment_spark("logs_10k.csv")

    spark_10k_results.append((execution_time, throughput))

    print(
        f"Run {run}: "
        f"Time = {execution_time:.6f} seconds, "
        f"Throughput = {throughput:.2f} records/second"
    )

Run 1: Time = 2.284840 seconds, Throughput = 4376.67 records/second
Run 2: Time = 1.989749 seconds, Throughput = 5025.76 records/second
Run 3: Time = 1.820078 seconds, Throughput = 5494.27 records/second
Run 4: Time = 1.140393 seconds, Throughput = 8768.91 records/second
Run 5: Time = 0.851974 seconds, Throughput = 11737.45 records/second


In [15]:
import statistics

# Sequential Python
python_times = [result[0] for result in python_10k_results]
python_throughputs = [result[1] for result in python_10k_results]

python_mean_time = statistics.mean(python_times)
python_std_time = statistics.stdev(python_times)
python_mean_throughput = statistics.mean(python_throughputs)

# Apache Spark
spark_times = [result[0] for result in spark_10k_results]
spark_throughputs = [result[1] for result in spark_10k_results]

spark_mean_time = statistics.mean(spark_times)
spark_std_time = statistics.stdev(spark_times)
spark_mean_throughput = statistics.mean(spark_throughputs)

print("Sequential Python")
print(f"Mean Execution Time: {python_mean_time:.6f} seconds")
print(f"Standard Deviation: {python_std_time:.6f}")
print(f"Mean Throughput: {python_mean_throughput:.2f} records/second")

print("\nApache Spark")
print(f"Mean Execution Time: {spark_mean_time:.6f} seconds")
print(f"Standard Deviation: {spark_std_time:.6f}")
print(f"Mean Throughput: {spark_mean_throughput:.2f} records/second")

Sequential Python
Mean Execution Time: 0.053982 seconds
Standard Deviation: 0.004789
Mean Throughput: 186359.43 records/second

Apache Spark
Mean Execution Time: 1.617407 seconds
Standard Deviation: 0.599711
Mean Throughput: 7080.61 records/second
